Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

In [18]:
!pip3 install pyspark~=4.0.0

In [19]:
import os
import sys
import pyspark.sql.functions as F
from pyspark.sql import Row
from pyspark.sql import SparkSession

In [21]:
spark = SparkSession.builder.getOrCreate()
spark

In [22]:
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml

--2026-04-03 11:41:26--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/posts_sample.xml
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74162295 (71M) [text/plain]
Saving to: ‘posts_sample.xml.2’

posts_sample.xml.2  100%[===================>]  70.73M   597KB/s    in 3m 41s  

2026-04-03 11:45:09 (327 KB/s) - ‘posts_sample.xml.2’ saved [74162295/74162295]



In [23]:
postsData = spark.read.format('xml') \
    .option('rowTag', 'row') \
    .option("timestampFormat", 'y/M/d H:m:s') \
    .load('posts_sample.xml')

In [24]:
print("Схема")
postsData.printSchema()

print("Первые 5 элементов")
postsData.show(n = 5)

print("Описание датасета")
postsData.describe().show()

print("Количество элементов")
print(postsData.count())

Схема
root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: string (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: string (nullable = true)
 |-- _CreationDate: string (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: string (nullable = true)
 |-- _LastEditDate: string (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)

Первые 5 элементов
+-----------------+------------+--------------------+-----------+--

In [25]:
# Ищем с 1 января 2010 года до 31 декабря 2020 года
dates = ("2010-01-01",  "2020-12-31")

# Выбираем посты в заданный промежуток
posts_by_date = postsData.filter(F.col("_CreationDate").between(*dates))
posts_by_date.show(10)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|_Tags|_Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|             NULL|        NULL|<p>No. (And more ...|       NULL

In [26]:
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv

--2026-04-03 11:45:36--  https://git.ai.ssau.ru/tk/big_data/raw/branch/master/data/programming-languages.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40269 (39K) [text/plain]
Saving to: ‘programming-languages.csv.1’

programming-languag 100%[===================>]  39.33K   119KB/s    in 0.3s    

2026-04-03 11:45:37 (119 KB/s) - ‘programming-languages.csv.1’ saved [40269/40269]



In [27]:
languagesData = spark.read.format('csv').option('header', 'true').option("inferSchema", True).load('programming-languages.csv').dropna()

In [28]:
print("Схема")
languagesData.printSchema()

print("Первые 5 элементов")
languagesData.show(n = 5)

print("Описание датасета")
languagesData.describe().show()

print("Количество элементов")
print(languagesData.count())

Схема
root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)

Первые 5 элементов
+----------+--------------------+
|      name|       wikipedia_url|
+----------+--------------------+
|   A# .NET|https://en.wikipe...|
|A# (Axiom)|https://en.wikipe...|
|A-0 System|https://en.wikipe...|
|        A+|https://en.wikipe...|
|       A++|https://en.wikipe...|
+----------+--------------------+
only showing top 5 rows
Описание датасета
+-------+--------+--------------------+
|summary|    name|       wikipedia_url|
+-------+--------+--------------------+
|  count|     699|                 699|
|   mean|    NULL|                NULL|
| stddev|    NULL|                NULL|
|    min|@Formula|https://en.wikipe...|
|    max|xHarbour|https://en.wikipe...|
+-------+--------+--------------------+

Количество элементов
699


In [29]:
# Названия языков
language_names = [str(x[0]) for x in languagesData.collect()]
language_names

['A# .NET',
 'A# (Axiom)',
 'A-0 System',
 'A+',
 'A++',
 'ABAP',
 'ABC',
 'ABC ALGOL',
 'ABSET',
 'ABSYS',
 'ACC',
 'Accent',
 'Ace DASL',
 'ACL2',
 'ACT-III',
 'Action!',
 'ActionScript',
 'Ada',
 'Adenine',
 'Agda',
 'Agilent VEE',
 'Agora',
 'AIMMS',
 'Alef',
 'ALF',
 'ALGOL 58',
 'ALGOL 60',
 'ALGOL 68',
 'ALGOL W',
 'Alice',
 'Alma-0',
 'AmbientTalk',
 'Amiga E',
 'AMOS',
 'AMPL',
 'Apex (Salesforce.com)',
 'APL',
 "App Inventor for Android's visual block language",
 'AppleScript',
 'Arc',
 'ARexx',
 'Argus',
 'AspectJ',
 'Assembly language',
 'ATS',
 'Ateji PX',
 'AutoHotkey',
 'Autocoder',
 'AutoIt',
 'AutoLISP / Visual LISP',
 'Averest',
 'AWK',
 'Axum',
 'B',
 'Babbage',
 'Bash',
 'BASIC',
 'bc',
 'BCPL',
 'BeanShell',
 'Batch (Windows/Dos)',
 'Bertrand',
 'BETA',
 'Bigwig',
 'Bistro',
 'BitC',
 'BLISS',
 'Blockly',
 'BlooP',
 'Blue',
 'Boo',
 'Boomerang',
 'Bourne shell (including',
 'bash and',
 'ksh )',
 'BREW',
 'BPEL',
 'C',
 'C--',
 'C++ – ISO/IEC 14882',
 'C# – ISO/IEC

In [30]:
# Функция позволяет определить какой язык содержится в теге поста
def includes_name(x):
    tag = None
    for name in language_names:
        n = '<' + name.lower() + '>'
        if n in str(x._Tags).lower():
            tag = name
            break
    if tag is None:
        tag = 'No'

    return (x[6], tag)


In [31]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# Создаем broadcast переменную для списка языков
language_names_broadcast = spark.sparkContext.broadcast(set(language_names))

# Функция для извлечения языка из тегов
def extract_language(tags):
    if tags is None:
        return None
    tags_lower = tags.lower()
    for name in language_names_broadcast.value:
        if '<' + name.lower() + '>' in tags_lower:
            return name
    return None

# Регистрируем UDF
extract_language_udf = F.udf(extract_language, StringType())

# Добавляем колонки с годом и языком
posts_with_lang = (posts_by_date
    .withColumn('Year', F.year('_CreationDate'))
    .withColumn('Language', extract_language_udf(F.col('_Tags')))
    .filter(F.col('Language').isNotNull()))

# Группируем по году и языку
language_counts = (posts_with_lang
    .groupBy('Year', 'Language')
    .agg(F.count('*').alias('Count')))

# Оконная функция для получения топ-10 по каждому году
window_spec = Window.partitionBy('Year').orderBy(F.desc('Count'))

# Получаем топ-10 языков для каждого года
top_10_by_year = (language_counts
    .withColumn('rank', F.row_number().over(window_spec))
    .filter(F.col('rank') <= 10)
    .drop('rank')
    .orderBy('Year', F.desc('Count')))

# Выводим результат
print("Топ-10 языков программирования по годам (2010-2020):")
top_10_by_year.show(20, truncate=False)

# Сохраняем в Parquet
output_path = "top_10_languages_2010_2020.parquet"
top_10_by_year.write.mode('overwrite').parquet(output_path)

print(f"\nРезультат сохранен в {output_path}")
print(f"Всего записей: {top_10_by_year.count()}")

Топ-10 языков программирования по годам (2010-2020):
+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2010|Java       |51   |
|2010|JavaScript |44   |
|2010|PHP        |43   |
|2010|Python     |24   |
|2010|Objective-C|22   |
|2010|C          |20   |
|2010|Ruby       |11   |
|2010|Delphi     |8    |
|2010|R          |3    |
|2010|AppleScript|3    |
|2011|PHP        |96   |
|2011|Java       |90   |
|2011|JavaScript |83   |
|2011|Python     |34   |
|2011|Objective-C|33   |
|2011|C          |24   |
|2011|Ruby       |19   |
|2011|Delphi     |8    |
|2011|Perl       |8    |
|2011|Bash       |7    |
+----+-----------+-----+
only showing top 20 rows

Результат сохранен в top_10_languages_2010_2020.parquet
Всего записей: 100
